# Ejercicio: Web Scraping

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [2]:
from bs4 import BeautifulSoup

file = 'rotisserie-chicken.html'

# Load the HTML file
with open(file, "r", encoding="utf-8") as file:
    html_content = file.read()
    
# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

In [3]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Rotisserie Chicken'

In [4]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [5]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Rotisserie Chicken
Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper
Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — b

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [6]:
# Find all the links to other recipes
recipe_links = soup.find_all("a", href=True)

# Filter and print only the links that are likely to be recipes
recipe_urls = []
for link in recipe_links:
    href = link['href']
    if "recipe" in href:
        recipe_urls.append(href)

# Print the recipe URLs
print("Linked Recipes:")
for url in recipe_urls:
    print(url)

Linked Recipes:
https://www.allrecipes.com/authentication/login?regSource=3675&relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
/account/add-recipe
https://www.myrecipes.com/favorites
https://www.allrecipes.com/authentication/logout?relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
https://www.magazines.com/allrecipes-magazine.html?utm_source=allrecipes.com&utm_medium=owned&utm_campaign=i111arr1w2661
https://www.magazines.com/allrecipes-magazine.html
https://www.allrecipes.com/recipes/17562/dinner/
https://www.allrecipes.com/recipes/17057/everyday-cooking/more-meal-ideas/5-ingredients/main-dishes/
https://www.allrecipes.com/recipes/15436/everyday-cooking/one-pot-meals/
https://www.allrecipes.com/recipes/1947/everyday-cooking/quick-and-easy/
https://www.allrecipes.com/recipes/455/everyday-cooking/more-meal-ideas/30-minute-meals/
https://www.allrecipes.com/recipes/17889/everyday-cooking/family-friendly/family-dinners/
https://www.allrecipes.com/recipes/94/soups-s

## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

La celda de codigo anterior muestra todos los links que contienen "recipe", pero la mayoría son links de navegación del sitio (categorías, menús, login). Las recetas reales en AllRecipes siguen el patrón `/recipe/NUMERO/nombre/`, así que usamos una expresion regular para filtrar solo esas. También eliminamos la receta original porque ya la tenemos parseada.

In [7]:
import re

recipe_urls_clean = set()
for link in recipe_links:
    href = link['href']
    if re.match(r'https://www\.allrecipes\.com/recipe/\d+/', href):
        recipe_urls_clean.add(href)

# Quitar la receta original porque ya la tenemos
recipe_urls_clean.discard("https://www.allrecipes.com/recipe/93168/rotisserie-chicken/")

recipe_urls_clean = list(recipe_urls_clean)
print(f"Recetas encontradas: {len(recipe_urls_clean)}")
for url in recipe_urls_clean:
    print(url)

Recetas encontradas: 16
https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/
https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/
https://www.allrecipes.com/recipe/8998/darn-good-chicken/
https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever/
https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/
https://www.allrecipes.com/recipe/19944/drunk-chicken/
https://www.allrecipes.com/recipe/34957/easy-barbeque-chicken/
https://www.allrecipes.com/recipe/281255/smoked-whole-chicken/
https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/
https://www.allrecipes.com/recipe/264278/miso-honey-chicken/
https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/
https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/
https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/
https://www.allrecipes.com/recipe/275044/grilled-chicken-under-a-brick/
https://www.allrecipes.com/recipe/214618/beer-c

Definimos una funcion reutilizable que extrae los campos de cualquier receta de AllRecipes. Usamos la misma logica de la Parte 2 pero con chequeos por si algún campo no existe en la pagina. Agregamos un User-Agent en los headers para que el servidor no rechace las peticiones, y un delay de 2 segundos entre requests para no sobrecargar el sitio.

In [8]:
import requests
import time

def extraer_receta(soup):
    titulo = soup.find("meta", {"property": "og:title"})
    titulo = titulo["content"] if titulo else "Sin titulo"
    
    desc = soup.find("meta", {"name": "description"})
    desc = desc["content"] if desc else ""
    
    ings = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
    ingredientes = [ing.get_text().strip() for ing in ings]
    
    pasos = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
    instrucciones = [p.get_text().strip() for p in pasos]
    
    return {
        "titulo": titulo,
        "descripcion": desc,
        "ingredientes": ingredientes,
        "instrucciones": instrucciones
    }

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

corpus = []

In [ ]:
# agregamos la receta original
corpus.append(extraer_receta(soup))
print(f"[0] {corpus[0]['titulo']}")

[0] Rotisserie Chicken


In [10]:
for i, url in enumerate(recipe_urls_clean):
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        sp = BeautifulSoup(resp.text, "html.parser")
        receta = extraer_receta(sp)
        receta["url"] = url
        corpus.append(receta)
        print(f"[{i+1}] {receta['titulo']}")
    except Exception as e:
        print(f"[{i+1}] Error con {url}: {e}")
    time.sleep(2)

print(f"\nTotal recetas en corpus: {len(corpus)}")

[1] Grilled Spatchcocked Chicken
[2] Smoked Beer Butt Chicken
[3] Darn Good Chicken
[4] The Best Beer Can Chicken Ever
[5] Cilantro-Lime Grilled Chicken
[6] Drunk Chicken
[7] Easy Barbeque Chicken
[8] Smoked Whole Chicken
[9] Best Beer Can Chicken
[10] Miso Honey Chicken
[11] Buttermilk Barbecue Chicken
[12] Good Frickin’ Paprika Chicken
[13] Rosemary Buttermilk Chicken
[14] Grilled Chicken Under a Brick
[15] Beer Can Chicken
[16] Beer Butt Chicken

Total recetas en corpus: 17


Para poder hacer retrieval, necesitamos representar cada receta como un solo bloque de texto. Concatenamos titulo, descripcion, ingredientes e instrucciones en un string por receta. Este sera el input para el modelo de embeddings.

In [11]:
documentos = []
for receta in corpus:
    texto = f"Receta: {receta['titulo']}\n"
    texto += f"Descripcion: {receta['descripcion']}\n"
    texto += f"Ingredientes: {', '.join(receta['ingredientes'])}\n"
    texto += f"Instrucciones: {' '.join(receta['instrucciones'])}"
    documentos.append(texto)

print(f"Documentos: {len(documentos)}")
print("---")
print(documentos[0][:500])

Documentos: 17
---
Receta: Rotisserie Chicken
Descripcion: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredientes: 1 (3 pound) whole chicken, 1 pinch salt, ¼ cup butter, melted, 1 tablespoon salt, 1 tablespoon ground paprika, ¼ tablespoon ground black pepper
Instrucciones: Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'l


Usamos el modelo `all-MiniLM-L6-v2` de sentence-transformers para generar embeddings de cada documento. 

In [12]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(documentos, show_progress_bar=True)

print(f"Shape: {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Shape: (17, 384)


La función de busqueda codifica la query del usuario con el mismo modelo, calcula similitud coseno contra todos los documentos del corpus, y retorna los top-k mas relevantes.

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

def buscar(query, top_k=3):
    query_emb = model.encode([query])
    sims = cosine_similarity(query_emb, embeddings)[0]
    indices = np.argsort(sims)[::-1][:top_k]
    
    resultados = []
    for idx in indices:
        resultados.append({
            "titulo": corpus[idx]["titulo"],
            "score": sims[idx],
            "texto": documentos[idx]
        })
    return resultados

Probamos el retrieval con una query sobre cerveza para verificar que las recetas más relevantes aparecen primero

In [14]:
resultados = buscar("chicken with beer")
for r in resultados:
    print(f"{r['score']:.4f} - {r['titulo']}")

0.5918 - Drunk Chicken
0.5678 - Beer Can Chicken
0.5671 - Beer Butt Chicken


Finalmente implementamos RAG, tomamos los documentos mas relevantes de la busqueda y los pasamos como contexto a Gemini para que genere una respuesta basada unicamente en las recetas del corpus. 

In [ ]:
API_KEY = ""

In [37]:
import google.generativeai as genai

genai.configure(api_key=API_KEY)
llm = genai.GenerativeModel("gemini-3.5-flash")

In [38]:
def rag(query, top_k=3):
    resultados = buscar(query, top_k)
    
    contexto = ""
    for r in resultados:
        contexto += f"--- {r['titulo']} (score: {r['score']:.4f}) ---\n"
        contexto += r["texto"] + "\n\n"
    
    prompt = f"""Basandote SOLO en las siguientes recetas, responde la pregunta del usuario.

{contexto}

Pregunta: {query}
Respuesta:"""
    
    respuesta = llm.generate_content(prompt)
    return respuesta.text, resultados

In [39]:
respuesta, docs = rag("How do I make chicken with beer?")
print("Documentos usados:")
for d in docs:
    print(f"  {d['score']:.4f} - {d['titulo']}")
print(f"\nRespuesta:\n{respuesta}")

Documentos usados:
  0.7256 - Beer Can Chicken
  0.7224 - Drunk Chicken
  0.7190 - Beer Butt Chicken

Respuesta:
Based on the provided recipes, there are three ways to make chicken with beer on a grill. All of these methods involve placing a whole chicken upright over a half-full can of beer:

### Method 1: Beer Can Chicken (Medium-High Heat)
1. **Prepare the rub:** In a small bowl, mix ⅓ cup brown sugar, 2 tablespoons chili powder, 2 tablespoons paprika, 2 teaspoons dry mustard, ½ teaspoon salt, and ¼ teaspoon ground black pepper.
2. **Assemble:** Place a half-full (12 fluid ounce) can of beer on a plate. Fit a 3-pound whole chicken upright over the can. Sprinkle 1 teaspoon of the seasoning mix into the top cavity of the chicken and rub the remaining seasoning over the entire surface.
3. **Cook:** Preheat a charcoal grill to medium-high heat (about 375°F / 190°C). Push the coals to the sides of the grill to create indirect heat. Place the chicken, standing on the can, over indirect he

In [40]:
respuesta2, docs2 = rag("Which recipe uses honey?")
print("Documentos usados:")
for d in docs2:
    print(f"  {d['score']:.4f} - {d['titulo']}")
print(f"\nRespuesta:\n{respuesta2}")

Documentos usados:
  0.4561 - Miso Honey Chicken
  0.4497 - Darn Good Chicken
  0.3594 - The Best Beer Can Chicken Ever

Respuesta:
Based on the provided recipes, the recipes that use honey are:

1. **Miso Honey Chicken** (uses 2 tablespoons of honey)
2. **Darn Good Chicken** (uses ½ cup of honey, warmed slightly)


**Prueba de fidelidad al contexto**

Una ventaja de RAG es que el modelo responde solo con lo que tiene en el corpus. Si preguntamos algo que no está en las recetas scrapeadas, debería indicar que no tiene esa información en lugar de inventar.

In [44]:
respuesta3, docs3 = rag("How do I make sushi?")
print("Documentos usados:")
for d in docs3:
    print(f"  {d['score']:.4f} - {d['titulo']}")
print(f"\nRespuesta:\n{respuesta3}")

Documentos usados:
  0.4211 - Good Frickin’ Paprika Chicken
  0.3850 - Miso Honey Chicken
  0.3677 - Grilled Chicken Under a Brick

Respuesta:
Basándome únicamente en las recetas proporcionadas, no hay información sobre cómo hacer sushi, ya que todas las recetas son de platos de pollo (Good Frickin’ Paprika Chicken, Miso Honey Chicken y Grilled Chicken Under a Brick).


In [45]:
respuesta4, docs4 = rag("What is a good vegetarian pasta recipe?")
print("Documentos usados:")
for d in docs4:
    print(f"  {d['score']:.4f} - {d['titulo']}")
print(f"\nRespuesta:\n{respuesta4}")

Documentos usados:
  0.4210 - Good Frickin’ Paprika Chicken
  0.4058 - Easy Barbeque Chicken
  0.3964 - Buttermilk Barbecue Chicken

Respuesta:
Basándome únicamente en las recetas proporcionadas, no hay ninguna receta de pasta vegetariana. Las únicas recetas disponibles son de pollo: *Good Frickin’ Paprika Chicken*, *Easy Barbeque Chicken* y *Buttermilk Barbecue Chicken*.


### Resumen del corpus construido

In [42]:
print("Resumen del corpus")
print(f"Total recetas: {len(corpus)}")
total_ings = sum(len(r['ingredientes']) for r in corpus)
print(f"Ingredientes totales: {total_ings} (promedio {total_ings/len(corpus):.1f} por receta)")
total_pasos = sum(len(r['instrucciones']) for r in corpus)
print(f"Pasos totales: {total_pasos} (promedio {total_pasos/len(corpus):.1f} por receta)")
print(f"Dimensión embeddings: {embeddings.shape}")

Resumen del corpus
Total recetas: 17
Ingredientes totales: 137 (promedio 8.1 por receta)
Pasos totales: 118 (promedio 6.9 por receta)
Dimensión embeddings: (17, 384)
